# Comparing pretrained VGG13, VGG16 and VGG19 on CUB-200-2011

**Goal:** find out how well three pretrained CNNs can name 200 bird species, and whether a deeper VGG is worth its extra cost.

**How to use this notebook:** run it from top to bottom. Nothing slow runs by default: training and the test evaluation only run if you turn on their switches in section 2. Captions are not used anywhere here.

The whole experiment, with the shape of the data at each step (batch size 8):

```
images on disk
   |  src/data/loaders.py  (the team's pipeline, unchanged)
   v
batch of images            [8, 3, 224, 224]     8 images, 3 colours, 224 x 224 pixels
   |  CNN encoder          (VGG conv blocks + two 4096-d layers)
   v
feature vector             [8, 4096]            a numeric summary of each image
   |  final layer          nn.Linear(4096, 200)
   v
logits                     [8, 200]             one score per bird species
   |  loss -> gradients -> optimizer
   v
updated weights, and the best version is saved as a checkpoint
```

## 1. The key ideas

| Idea | What it means | Bird example |
|---|---|---|
| **CNN encoder** | The convolution layers that turn pixels into useful features. Early layers find edges and colours; deep layers find parts. | It notices "red patch on the head" and "long thin beak". It does not know any species names. |
| **Classification head** | The layers after the encoder that turn features into one score per class. | It learns that a red head plus a long beak points to a certain woodpecker. |
| **Pretrained weights** | Weights already learned on another task, here 1.2 million ImageNet photos of 1,000 kinds of object. | Someone who has looked at many photos already knows what feathers and beaks look like. |
| **Transfer learning** | Reusing those weights for a new task instead of starting from random numbers. | With only about 25 training photos per species, this is a big advantage. |
| **Freezing layers** | Blocking updates to some layers, so they keep their current weights. | The encoder stays as it is while the new head learns. |
| **Fine-tuning** | Unfreezing and training everything with very small steps. | The encoder now adjusts slightly, from general objects towards small differences between birds. |
| **Training loss** | How wrong the model is on images it learns from. Lower is better. | Marks on practice questions it has already seen. |
| **Validation accuracy** | Share of correct answers on held-back images, used to make decisions such as which epoch to keep. | A mock exam, taken many times. |
| **Test accuracy** | Share of correct answers on images never used for any decision. This is the number you report. | The real exam, taken once at the very end. |

**Why three datasets?** If you choose the best model by looking at the test set, the test set has influenced your choice, and its score is no longer a fair estimate. So: train on training images, choose on validation images, and report on test images.

## Colab setup

Run on a Colab GPU runtime (*Runtime > Change runtime type > T4 GPU*). This cell checks for the GPU, clones the repository (code and images) into `/content` and installs the requirements. It does nothing outside Colab.

The repository is private, so add a GitHub token as a Colab secret named `GITHUB_TOKEN`. If there is no secret, the cell asks for the token instead.

In [4]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO = "omarsaher/image-classfication-and-caption"
BRANCH = "feature/vgg-cnn-encoder"
WORKDIR = "/content/image-classfication-and-caption"

if importlib.util.find_spec("google.colab"):
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("No GPU found. Use Runtime > Change runtime type > T4 GPU, then run again.")
    print("GPU:", torch.cuda.get_device_name(0))

    if not Path(WORKDIR).exists():
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            from getpass import getpass
            token = getpass("GitHub token: ")

        # Clone the private repo; the token is removed from the remote URL right after
        result = subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH,
             f"https://{token}@github.com/{REPO}.git", WORKDIR],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError("git clone failed:\n" + result.stderr.replace(token, "***"))
        subprocess.run(["git", "-C", WORKDIR, "remote", "set-url", "origin",
                        f"https://github.com/{REPO}.git"], check=True)

    os.chdir(WORKDIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

GPU: Tesla T4


RuntimeError: git clone failed:
***C***l***o***n***i***n***g*** ***i***n***t***o*** ***'***/***c***o***n***t***e***n***t***/***i***m***a***g***e***-***c***l***a***s***s***f***i***c***a***t***i***o***n***-***a***n***d***-***c***a***p***t***i***o***n***'***.***.***.***
***f***a***t***a***l***:*** ***c***o***u***l***d*** ***n***o***t*** ***r***e***a***d*** ***P***a***s***s***w***o***r***d*** ***f***o***r*** ***'***h***t***t***p***s***:***/***/***g***i***t***h***u***b***.***c***o***m***'***:*** ***N***o*** ***s***u***c***h*** ***d***e***v***i***c***e*** ***o***r*** ***a***d***d***r***e***s***s***
***

In [ ]:
import os
import sys
from pathlib import Path

# Config paths are relative to the repository root, so go there first,
# wherever Jupyter was started from.
REPO_ROOT = next(
    folder for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "configs" / "vgg_classification.yaml").is_file()
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import torch
from IPython.display import Image, display

from src.data.loaders import build_train_val_loaders
from src.evaluation.vgg_report import compare_models, evaluate_vgg
from src.models.vgg_classifier import (
    build_vgg_classifier,
    freeze_feature_extractor,
    parameter_counts,
    unfreeze_feature_extractor,
)
from src.training.train_vgg import load_vgg_config, run_dir, train_vgg, validate_data_paths
from src.training.vgg_smoke_test import run_smoke_test
from src.utils.seed import get_device

print("Repository:", REPO_ROOT)
print("PyTorch:", torch.__version__, "| device:", get_device())


def show_figure(path):
    # The team's plotting code saves figures to files, so we display the file
    path = Path(path)
    if path.is_file():
        display(Image(filename=str(path)))
    else:
        print(f"Not created yet: {path}")

## 2. Switches

Both start as `False`, so running the whole notebook is safe. Turn them on when you are ready.

- `RUN_TRAINING` trains every model in `MODELS_TO_TRAIN`. This takes a long time on a CPU.
- `RUN_TEST_EVALUATION` evaluates finished models on the test set. The first time, it downloads the test set (about 0.56 GB).
- `OVERWRITE` allows replacing a saved result. Without it, training stops instead of replacing a finished checkpoint.

In [ ]:
RUN_TRAINING = False
RUN_TEST_EVALUATION = False
OVERWRITE = False
MODELS_TO_TRAIN = ["vgg13", "vgg16", "vgg19"]

## 3. Configuration

| | |
|---|---|
| **What it does** | Keeps every setting in one file, `configs/vgg_classification.yaml`. |
| **Why we need it** | You can change an experiment without editing code, and each run saves the settings it used. |
| **What goes in** | The YAML file. Its `data` section is placed on top of the team's `cls_scratch.yaml`, so the split, the statistics and the folders are not copied. |
| **What comes out** | A nested Python dict, `cfg`. |
| **What comes next** | Every other step reads from `cfg`. |

To change a value for one session, pass overrides: `load_vgg_config(path, ["data.batch_size=4"])`.

In [ ]:
cfg = load_vgg_config(REPO_ROOT / "configs" / "vgg_classification.yaml")
validate_data_paths(cfg)   # stops here if a file is missing, before anything can create a new split

print("Models:        ", cfg["model"]["architectures"])
print("Batch size:    ", cfg["data"]["batch_size"], "| workers:", cfg["data"]["num_workers"])
print("Validation:    ", cfg["data"]["split_file"])
print("Normalisation: ", cfg["data"]["stats_file"])
print("Phase 1 (head):", cfg["phases"]["head"])
print("Phase 2 (fine):", cfg["phases"]["finetune"])

## 4. Data pipeline (the team's code, unchanged)

| | |
|---|---|
| **What it does** | `build_train_val_loaders` reads the metadata, applies the saved split, and returns loaders that deliver batches of images and labels. |
| **Why we need it** | Every experiment in the project uses the same images and the same split, so results can be compared fairly. |
| **What goes in** | The `data` section of `cfg`. |
| **What comes out** | `train_loader`, `val_loader`, 200 class names, and the mean and std used for normalisation. |
| **What comes next** | Batches go into the model. |

Things this pipeline guarantees:

- **Validation:** 5 original images per species (1,000), read from `splits/val_original_ids.json`. A new random split is never created while that file exists.
- **No leakage:** the split is grouped by `original_id`, so a validation photo cannot reach training through an augmented copy.
- **Clean validation images:** the validation loader only resizes and centre-crops. There are no random changes, as the printout below shows.
- **Test set:** only `evaluate_vgg` loads it, and only when you ask.

**A note on normalisation.** The team's pipeline uses statistics computed from its own training images, while the VGG weights were trained with ImageNet's. They are close but not identical (green mean 0.501 vs 0.456, blue std 0.262 vs 0.225). The team's values are kept here so that this experiment stays consistent with the rest of the project. Testing ImageNet's values would be a reasonable later experiment.

In [ ]:
train_loader, val_loader, class_names, (mean, std) = build_train_val_loaders(cfg)

print(f"Training images:   {len(train_loader.dataset):,}")
print(f"Validation images: {len(val_loader.dataset):,}")
print(f"Classes:           {len(class_names)}  e.g. {class_names[:3]}")
print(f"Batches per epoch: {len(train_loader)}")
print("Mean:", [round(v, 3) for v in mean], "| std:", [round(v, 3) for v in std])
print("\nTraining transform (random changes, a new version of each image every epoch):")
print(train_loader.dataset.transform)
print("\nValidation transform (the same result every time):")
print(val_loader.dataset.transform)

## 5. The model: encoder + head

**CNN encoder** (`src/models/encoders/vgg.py`)

| | |
|---|---|
| **What it does** | Runs the official torchvision VGG layers and produces a feature vector. |
| **Why we need it** | Raw pixels say little about species. These layers already know how to extract useful visual features. |
| **What goes in** | Images, `[batch, 3, 224, 224]`. |
| **What comes out** | Features, `[batch, 4096]`. |
| **What comes next** | The new final layer. |

**Final layer** (`nn.Linear(4096, 200)`, added by the team's `ImageClassifier`)

| | |
|---|---|
| **What it does** | Turns the 4096 features into one score (logit) per species. |
| **Why we need it** | The original layer scores 1,000 ImageNet objects, and we need 200 birds. It starts from random weights. |
| **What goes in** | Features, `[batch, 4096]`. |
| **What comes out** | Logits, `[batch, 200]`. The highest score is the prediction. |
| **What comes next** | The loss compares the logits with the true labels. |

**What "freezing the feature extractor" covers.** The convolution blocks are frozen. The two pretrained 4096-d layers stay trainable along with the new layer. Together, those three layers are what we call the classification head.

**VGG13, VGG16 and VGG19 differ only in their number of convolution layers** (10, 13 and 16). Their heads are identical, which you can see below: phase 1 trains the same number of parameters for all three.

We keep the official classifier, as requested. Its trade-off is size: about 89% of VGG16's parameters are in those fully connected layers, because flattening a 512 x 7 x 7 feature map into a 4096-d layer alone takes 103 million weights. Global average pooling would shrink the model a lot, but it would no longer be the official architecture and would throw away the pretrained 4096-d weights.

In [ ]:
# pretrained=False needs no download: this cell only looks at the structure
rows = []
for arch in cfg["model"]["architectures"]:
    model = build_vgg_classifier(arch, pretrained=False)
    total = parameter_counts(model)["total"]
    freeze_feature_extractor(model)
    frozen_trainable = parameter_counts(model)["trainable"]
    unfreeze_feature_extractor(model)
    conv_layers = sum(isinstance(layer, torch.nn.Conv2d) for layer in model.encoder.features)
    rows.append({
        "model": arch,
        "conv layers": conv_layers,
        "total parameters": total,
        "trainable when frozen": frozen_trainable,
        "conv parameters (frozen)": total - frozen_trainable,
    })
    if arch == "vgg16":
        print("VGG16 fully connected layers:\n", model.encoder.fc)
        print("New head:\n", model.head)
    del model

display(pd.DataFrame(rows))

## 6. Smoke test

A smoke test is a quick check that everything is connected correctly, before you spend hours training. For each model, on one real batch, `src/training/vgg_smoke_test.py` checks that:

1. a batch loads from the team's pipeline;
2. the model accepts it and outputs `[batch_size, 200]`;
3. the loss is a finite number;
4. one training step runs;
5. in phase 1, the final layer changes while a frozen convolution layer stays exactly the same;
6. in phase 2, the convolution layer changes too.

If a check fails, the cell stops with an error naming it.

The first run downloads the pretrained weights (about 1.6 GB in total) into PyTorch's cache. After that they are reused and not downloaded again.

The first loss should be close to `ln(200) = 5.3`, which is what a model that is guessing would get. Passing this test shows the code works. It says nothing about accuracy yet.

In [ ]:
smoke_results = pd.DataFrame(run_smoke_test(cfg))

# A rough time estimate from one step. It leaves out validation and image loading,
# so treat it as a lower bound; a single step is also a noisy measurement.
head_epochs = cfg["phases"]["head"]["epochs"]
finetune_epochs = cfg["phases"]["finetune"]["epochs"]
smoke_results["est_hours_all_epochs"] = (
    (head_epochs * smoke_results["head_step_sec"] + finetune_epochs * smoke_results["finetune_step_sec"])
    * len(train_loader) / 3600
).round(1)
display(smoke_results)

## 7. Training

`train_vgg` (`src/training/train_vgg.py`) runs both phases for one model. It uses the team's `Trainer` for the actual loop, so all models in the project train the same way.

| | |
|---|---|
| **What it does** | Phase 1 freezes the convolution blocks and trains the head. Phase 2 reloads the best phase 1 weights, unfreezes everything, and continues with a learning rate 30 times smaller. |
| **Why two phases** | The new final layer starts random, so early gradients are large and noisy. Training it first stops that noise from damaging the good pretrained features. |
| **What goes in** | `cfg`, a model name, and the training and validation images. |
| **What comes out** | `checkpoints/<model>_best.pt`, `history.json`, `curves.png`, `train_summary.json`. |
| **What comes next** | Test evaluation loads the checkpoint. |

The parts inside the training loop:

| Part | Its job |
|---|---|
| **CrossEntropyLoss** | Gives one number for how wrong the logits are. **Label smoothing** (0.1) stops the model being rewarded for extreme certainty, which helps with species that look alike. |
| **AdamW** | Uses the gradients to update the weights. **Weight decay** keeps weights small, which reduces overfitting. |
| **Cosine scheduler** | Lowers the learning rate smoothly: bigger steps early, careful steps late. |
| **Validation each epoch** | Measures progress on images the model does not learn from. |
| **Checkpoint** | Saves the weights whenever validation accuracy reaches a new best, so a later decline does not lose them. |
| **Early stopping** | Stops a phase if validation accuracy has not improved for a set number of epochs. |
| **Mixed precision** | Speeds up GPU training. It is switched off automatically on a CPU. |

The final checkpoint is whichever phase reached the higher validation accuracy.

**To check the whole flow first**, run a tiny version that writes to a separate folder. Its results mean nothing, but it finds problems within minutes:

```python
quick_cfg = load_vgg_config(REPO_ROOT / "configs" / "vgg_classification.yaml", [
    "train.max_batches=3", "phases.head.epochs=1", "phases.finetune.epochs=1",
    "experiment.output_dir=outputs/vgg_quick",
    "experiment.checkpoint_dir=outputs/vgg_quick/checkpoints",
])
train_vgg(quick_cfg, "vgg13")
```

In [ ]:
if RUN_TRAINING:
    for arch in MODELS_TO_TRAIN:
        train_vgg(cfg, arch, overwrite=OVERWRITE)
else:
    print("RUN_TRAINING is False, so nothing was trained.")

### Reading the training curves

- **Both losses falling:** the model is learning.
- **Training accuracy far above validation accuracy:** overfitting. It is memorising training images. Some gap is normal with about 25 images per species.
- **A jump when phase 2 starts:** expected, because the convolution layers have just begun to learn.
- **Training accuracy lower than validation accuracy early on:** this can happen. Training images pass through dropout and random crops, and validation images do not.

In [ ]:
for arch in MODELS_TO_TRAIN:
    show_figure(run_dir(cfg, arch) / "curves.png")

## 8. Final evaluation on the test set

`evaluate_vgg` (`src/evaluation/vgg_report.py`) loads the best checkpoint and predicts every test image. Run it **only after all training and model choices are finished**. If you change your settings because of a test score, that score stops being a fair estimate.

| Metric | Question it answers |
|---|---|
| **Accuracy** | How often is the top prediction right? |
| **Top-5 accuracy** | How often is the right species among the 5 highest scores? Useful when species look very alike. |
| **Macro precision** | When the model names a species, how often is it right, averaged over species? |
| **Macro recall** | Of each species' images, how many does it find, averaged over species? |
| **Macro F1** | The balance between the two. "Macro" means every species counts equally. |

Each model gets a folder `eval_test/` with `metrics.json`, `predictions.csv`, a confusion matrix, and example predictions.

In [ ]:
if RUN_TEST_EVALUATION:
    for arch in MODELS_TO_TRAIN:
        evaluate_vgg(cfg, arch, split="test")
else:
    print("RUN_TEST_EVALUATION is False, so the test set was not loaded.")

In [ ]:
for arch in MODELS_TO_TRAIN:
    eval_dir = run_dir(cfg, arch) / "eval_test"
    show_figure(eval_dir / "example_predictions.png")
    show_figure(eval_dir / "confusion_matrix.png")

**Reading the confusion matrix:** rows are true species and columns are predictions. A bright diagonal means correct predictions. Bright spots away from the diagonal are species that get mixed up. CUB's classes are ordered so that related species sit together, so errors tend to cluster close to the diagonal. `metrics.json` lists the most confused pairs by name.

## 9. Comparing the models

`compare_models` combines each model's results into one table and saves `comparison_test.csv`, along with a test accuracy bar chart and a chart of the training curves.

Questions to ask of the table:

- Does a deeper model actually score higher, and by more than a small margin? With 5,794 test images, a difference of a few tenths of a percent is close to noise.
- Is any gain worth the extra training time?
- Did the model with the best **validation** accuracy also get the best test accuracy? Choose your final model by validation accuracy, and report its test accuracy.

`max_batches` should be empty and `n_images` should be 5794. Anything else means that row came from a quick check, not a real result.

In [ ]:
if RUN_TEST_EVALUATION:
    comparison = compare_models(cfg, MODELS_TO_TRAIN)
    display(comparison.round(4))

output_dir = Path(cfg["experiment"]["output_dir"])
show_figure(output_dir / "test_accuracy_comparison.png")
show_figure(output_dir / "training_comparison.png")